In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("emstatsl/csgo-cheating-dataset")

print("Path to dataset files:", path)

100%|██████████| 461M/461M [00:05<00:00, 85.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/emstatsl/csgo-cheating-dataset/versions/1


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Load the Dataset
# Assuming the kaggle dataset 'csgo_cheating_dataset.csv' is in the directory
# Features: delta_pitch, delta_yaw, angular_velocity, crosshair_offset, latency, snap_consistency
# Target: label (0 = Legitimate, 1 = Cheating)
try:
    df = pd.read_csv('csgo_cheating_dataset.csv')
except FileNotFoundError:
    print("Dataset not found. Creating synthetic data for demonstration...")
    data_size = 1000
    df = pd.DataFrame({
        'delta_pitch': np.random.normal(0.5, 0.2, data_size),
        'delta_yaw': np.random.normal(0.5, 0.2, data_size),
        'angular_velocity': np.random.uniform(0, 500, data_size),
        'crosshair_offset': np.random.uniform(0, 10, data_size),
        'latency': np.random.randint(50, 400, data_size),
        'snap_consistency': np.random.uniform(0.8, 1.0, data_size),
        'label': np.random.choice([0, 1], data_size)
    })

# 2. Preprocessing
# Defining our feature columns based on behavioral signatures
features = ['delta_pitch', 'delta_yaw', 'angular_velocity',
            'crosshair_offset', 'latency', 'snap_consistency']
X = df[features]
y = df['label']

# Splitting data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardizing features (important for behavioral metrics with different scales)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 3. Model Implementation
# Random Forest is chosen for its robustness and ability to prevent overfitting
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# 4. Evaluation
y_pred = model.predict(X_test)

print("--- Model Performance Metrics ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 5. Feature Importance Analysis
# This allows us to see which behavior (e.g., Snap Consistency) most indicates a cheater
importances = model.feature_importances_
feat_importances = pd.Series(importances, index=features)
print("\n--- Top Behavioral Indicators of Cheating ---")
print(feat_importances.sort_values(ascending=False))

Dataset not found. Creating synthetic data for demonstration...
--- Model Performance Metrics ---
Accuracy: 56.50%

Confusion Matrix:
[[58 41]
 [46 55]]

Classification Report:
              precision    recall  f1-score   support

           0       0.56      0.59      0.57        99
           1       0.57      0.54      0.56       101

    accuracy                           0.56       200
   macro avg       0.57      0.57      0.56       200
weighted avg       0.57      0.56      0.56       200


--- Top Behavioral Indicators of Cheating ---
crosshair_offset    0.178571
snap_consistency    0.177912
angular_velocity    0.176825
delta_pitch         0.163507
delta_yaw           0.158865
latency             0.144320
dtype: float64
